
##### ==================ROADMAP — FRAMEWORK DE DATA QUALITY==================

Este projeto tem como objetivo desenvolver um framework **genérico e reutilizável de Data Quality**, capaz de analisar diferentes DataFrames e fontes de dados sem depender de regras específicas de um determinado domínio.

A arquitetura foi organizada em etapas independentes, permitindo evoluir gradualmente da análise estrutural dos dados até a geração de um indicador consolidado de qualidade.

> **Princípio do projeto:** o framework deve ser capaz de analisar diferentes fontes de dados sem conhecer previamente o significado das colunas ou depender de regras específicas de negócio.

###### ETAPAS DO PROJETO

***01. SCHEMA PROFILE*** ✅  
Análise estrutural do DataFrame, contemplando tipos de dados, completude, valores NULL, valores vazios, valores distintos e cardinalidade.

***02. DISTRIBUTION PROFILE***  
Análise da distribuição dos valores, frequência, concentração, diversidade e comportamento predominante das colunas.

***03. PATTERN PROFILE***  
Identificação automática de padrões nos dados, como comprimento, estrutura, composição de caracteres e padrões predominantes, sem depender do significado da coluna.

***04. DUPLICITY PROFILE*** 
Análise de duplicidade e unicidade dos registros e identificação de possíveis comportamentos de repetição nos dados.

***05. CONSISTENCY PROFILE*** 
Análise da consistência entre valores e colunas, buscando identificar comportamentos contraditórios ou relações inconsistentes dentro do próprio dataset.

***06. OUTLIER PROFILE***  
Identificação de valores ou comportamentos estatisticamente fora do padrão esperado, utilizando técnicas de análise de distribuição e detecção de anomalias.

***07. CORRELATION PROFILE***  
Análise de relações, associações e possíveis dependências entre as variáveis disponíveis no DataFrame.

***08. DATA QUALITY SCORE***  
Consolidação dos indicadores produzidos nas etapas anteriores em uma visão geral de qualidade, permitindo acompanhar o nível de qualidade do dataset de forma objetiva.

###### EVOLUÇÃO DO FRAMEWORK

O desenvolvimento seguirá uma abordagem incremental:

***ESTRUTURA → DISTRIBUIÇÃO → PADRÕES → DUPLICIDADE → CONSISTÊNCIA → ANOMALIAS → RELACIONAMENTOS → SCORE***

Cada etapa deverá gerar informações que possam ser utilizadas como entrada para as etapas seguintes, mantendo o framework ***genérico, modular, reutilizável e independente da fonte de dados***.

###### COMENTÁRIOS NO SCRIPT

O comentário presente após o título de referência da célula segue a estruturação explicativa:

- O QUE FAZ:
- COMO FAZ:
- POR QUE É IMPORTANTE:
- PERGUNTA RESPONDIDA:


##### 1. Fonte de Dados e DataFrame

###### DataFrame — Ocorrências Emergenciais da Rede de Distribuição 2026

Para este projeto, foi utilizado um conjunto de dados disponibilizado pela ***ANEEL — Agência Nacional de Energia Elétrica***, contendo informações sobre ocorrências emergenciais na rede de distribuição de energia elétrica durante o ano de 2026.

O DataFrame utilizado nesta etapa serve como ***fonte de dados para demonstração e validação do framework de Data Quality***.

###### Arquitetura Genérica

A célula de carregamento dos dados foi desenvolvida de forma independente das etapas de análise. Dessa forma, a fonte de dados pode ser substituída sem a necessidade de alterar o código das etapas seguintes.

A proposta é que, em uma versão futura, a primeira célula seja substituída por uma estrutura genérica capaz de receber diferentes fontes de dados, como:

- Arquivos CSV;
- Arquivos Parquet;
- Tabelas Delta;
- Tabelas SQL;
- Volumes do Databricks;
- Outras fontes compatíveis com Apache Spark.

Após o carregamento, o DataFrame passa a ser utilizado como entrada para as etapas de ***Schema Profile, Data Quality, análise de padrões, relevância e correlação***, mantendo o restante do notebook independente da origem dos dados.

> ***Princípio do projeto:*** a fonte de dados deve ser substituível sem alterar a lógica de análise e qualidade.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import NumericType

In [0]:
# ============================================================
# 1. CARREGAMENTO DA FONTE DE DADOS
# ============================================================

# FONTE:
# Carrega o conjunto de dados de Ocorrências Emergenciais da
# Rede de Distribuição de Energia Elétrica disponibilizado pela ANEEL.

# OBJETIVO:
# Disponibilizar o DataFrame que será utilizado como entrada
# para as etapas de análise e avaliação de Data Quality.

# ARQUIVO:
# O dataset está armazenado em formato CSV no Volume do Databricks.

path = "/Volumes/setor_eletrico/distribuicao/ocorrencias_emergenciais_setor_eletrico/ocorrencias-emergenciais-rede-distribuicao-2026.csv"


# ============================================================
# 2. LEITURA DO DATASET
# ============================================================

# HEADER:
# Define a primeira linha do arquivo como nome das colunas.

# DELIMITER:
# Utiliza ponto e vírgula (;) como separador dos campos,
# conforme o formato original do arquivo.

# QUOTE:
# Define aspas duplas (") como delimitador de textos,
# permitindo interpretar corretamente valores que possam
# conter caracteres especiais ou separadores.

# INFERSCHEMA:
# Permite que o Spark identifique automaticamente os tipos
# de dados das colunas durante a leitura do arquivo.

df_ocorrencias_emergenciais_rede_distribuicao_2026 = (
    spark.read
    .option("header", "true")
    .option("delimiter", ";")
    .option("quote", '"')
    .option("inferSchema", "true")
    .csv(path)
)


# ============================================================
# 3. VALIDAÇÃO INICIAL
# ============================================================

# EXIBIÇÃO:
# Apresenta uma amostra dos dados para validar visualmente
# se o arquivo foi carregado corretamente antes de iniciar
# as análises de qualidade.

display(
    df_ocorrencias_emergenciais_rede_distribuicao_2026
)

##### 2. Schema Profile da Tabela — Análise Estrutural e Qualidade dos Dados

O ***Schema Profile*** é a primeira etapa do processo de análise de qualidade de dados. Seu objetivo é realizar uma avaliação estrutural do DataFrame, identificando características importantes de cada coluna antes da aplicação de regras de negócio ou análises estatísticas mais avançadas.

Nesta etapa são analisados aspectos como ***tipo de dado, completude, valores nulos, valores vazios, quantidade de valores distintos e cardinalidade***. Essas informações permitem identificar rapidamente possíveis problemas de preenchimento, colunas de baixa ou alta variabilidade e comportamentos que merecem investigação.

> ***Importante:*** o Schema Profile não determina sozinho se um dado está correto ou incorreto. Ele identifica padrões e comportamentos que servirão de base para as próximas etapas de Data Quality, como análise de padrões, validade, duplicidade, consistência, relevância e correlação.

---

###### Objetivos desta etapa

- Identificar a estrutura do DataFrame;
- Quantificar registros e colunas;
- Avaliar o preenchimento das informações;
- Identificar valores `NULL` e vazios;
- Medir a diversidade dos valores;
- Identificar colunas de baixa e alta cardinalidade;
- Detectar colunas que merecem investigação;
- Criar uma base estruturada para as próximas análises de qualidade.

###### Etapas realizadas:
- ***Schema da tabela***: mostra os tipos de dados de cada coluna, permitindo verificar se estão coerentes com o conteúdo esperado.  
- ***Resumo estatístico***: fornece métricas como mínimo, máximo, média e desvio padrão para colunas numéricas.  
- ***Contagem de nulos***: avalia a completude dos dados, identificando colunas com ausência de valores.  
- ***Cardinalidade***: mede a diversidade de valores em cada coluna, útil para detectar atributos pouco informativos ou com excesso de variação.  
- ***Distribuição de valores***: ajuda a visualizar padrões e identificar possíveis outliers ou inconsistências.  

Esse perfilamento inicial é essencial em ***engenharia de dados*** e ***qualidade da informação***, pois fornece insumos para:
- Definir regras de validação e limpeza.  
- Avaliar relevância das colunas para análises estatísticas ou modelos de machine learning.  
- Garantir integridade e consistência antes de avançar para etapas de transformação e modelagem.  


In [0]:
# ============================================================
# 1. VALIDAÇÃO DO DATAFRAME
# ============================================================

# O QUE FAZ:
# Valida se o DataFrame está disponível e possui colunas para que o processo de Data Quality possa ser executado.

# COMO FAZ:
# Verifica se o DataFrame foi definido e se possui pelo menos uma coluna disponível para análise.

# POR QUE É IMPORTANTE:
# Evita a execução das etapas seguintes sobre um DataFrame inexistente ou sem estrutura para análise.

# PERGUNTA RESPONDIDA:
# O DataFrame está pronto para ser analisado?

if df is None:
    raise ValueError("O DataFrame não foi definido.")

if len(df.columns) == 0:
    raise ValueError("O DataFrame não possui colunas.")

In [0]:
# ============================================================
# 2. INFORMAÇÕES GERAIS
# ============================================================

# O QUE FAZ:
# Obtém as informações básicas sobre o tamanho e a estrutura do DataFrame.

# COMO FAZ:
# Calcula a quantidade total de registros e a quantidade total de colunas existentes na tabela.

# POR QUE É IMPORTANTE:
# Define a dimensão do dataset e fornece a base necessária para os cálculos das métricas de qualidade.

# PERGUNTA RESPONDIDA:
# Qual é o tamanho e a estrutura básica do dataset?

total_registros = df.count()
total_colunas = len(df.columns)

In [0]:
# ============================================================
# 3. CONSTRUÇÃO DAS MÉTRICAS
# ============================================================

# O QUE FAZ:
# Define automaticamente as métricas que serão utilizadas para avaliar a qualidade de cada coluna.

# COMO FAZ:
# Percorre todas as colunas do DataFrame e calcula a quantidade de valores NULL, valores vazios e valores distintos.

# POR QUE É IMPORTANTE:
# Permite que o Schema Profile seja utilizado de forma genérica em diferentes tabelas sem depender de nomes de colunas previamente definidos.

# PERGUNTA RESPONDIDA:
# Quais características básicas de qualidade devem ser avaliadas em cada coluna?

metricas = []

for campo in df.schema.fields:

    nome_coluna = campo.name

    coluna = F.col(nome_coluna)
    coluna_string = coluna.cast("string")

    metricas.extend([

        # Valores NULL
        F.sum(
            F.when(coluna.isNull(), 1).otherwise(0)
        ).alias(f"{nome_coluna}__null"),

        # Valores vazios
        F.sum(
            F.when(
                coluna.isNotNull() &
                (F.trim(coluna_string) == ""),
                1
            ).otherwise(0)
        ).alias(f"{nome_coluna}__vazio"),

        # Valores distintos
        F.approx_count_distinct(
            coluna
        ).alias(f"{nome_coluna}__distintos")
    ])

In [0]:
# ============================================================
# 4. EXECUÇÃO DO PROFILE
# ============================================================

# O QUE FAZ:
# Executa as métricas definidas anteriormente sobre todos os registros do DataFrame.

# COMO FAZ:
# Realiza as agregações necessárias para obter os resultados das métricas de cada coluna.

# POR QUE É IMPORTANTE:
# Centraliza a execução das métricas e permite obter uma visão consolidada da qualidade estrutural da tabela.

# PERGUNTA RESPONDIDA:
# Quais são os resultados das métricas de qualidade calculadas para esta tabela?

resultado = df.agg(*metricas).collect()[0]

In [0]:
# ============================================================
# 5. CONSTRUÇÃO DO RESULTADO
# ============================================================

# O QUE FAZ:
# Transforma os resultados brutos das métricas em indicadores de qualidade mais fáceis de interpretar.

# COMO FAZ:
# Calcula a quantidade e o percentual de valores preenchidos, NULL, vazios, não preenchidos e distintos para cada coluna.

# POR QUE É IMPORTANTE:
# Permite comparar o comportamento das colunas utilizando indicadores absolutos e relativos, independentemente do tamanho da tabela.

# PERGUNTA RESPONDIDA:
# Qual é o nível de preenchimento e diversidade de cada coluna?

profile = []

for campo in df.schema.fields:

    nome_coluna = campo.name
    tipo_dado = campo.dataType.simpleString()

    qtd_null = resultado[f"{nome_coluna}__null"] or 0
    qtd_vazio = resultado[f"{nome_coluna}__vazio"] or 0
    qtd_distintos = resultado[f"{nome_coluna}__distintos"] or 0

    qtd_nao_preenchidos = qtd_null + qtd_vazio

    qtd_preenchidos = (
        total_registros - qtd_nao_preenchidos
    )

    pct_null = (
        qtd_null / total_registros * 100
        if total_registros > 0 else 0
    )

    pct_vazio = (
        qtd_vazio / total_registros * 100
        if total_registros > 0 else 0
    )

    pct_preenchido = (
        qtd_preenchidos / total_registros * 100
        if total_registros > 0 else 0
    )

    pct_distintos = (
        qtd_distintos / total_registros * 100
        if total_registros > 0 else 0
    )

    profile.append((
        nome_coluna,
        tipo_dado,
        total_registros,
        qtd_preenchidos,
        qtd_null,
        qtd_vazio,
        qtd_nao_preenchidos,
        qtd_distintos,
        round(pct_preenchido, 2),
        round(pct_null, 2),
        round(pct_vazio, 2),
        round(pct_distintos, 2)
    ))

In [0]:
# ============================================================
# 6. DATAFRAME DO SCHEMA PROFILE
# ============================================================

# O QUE FAZ:
# Cria um DataFrame consolidado contendo todas as métricas calculadas para cada coluna.

# COMO FAZ:
# Organiza os resultados do profile em uma estrutura tabular com uma linha para cada coluna analisada.

# POR QUE É IMPORTANTE:
# O schema_profile se torna a principal fonte para as análises, visualizações e classificações realizadas nas etapas seguintes.

# PERGUNTA RESPONDIDA:
# Como está a qualidade estrutural de cada coluna?

schema_profile = spark.createDataFrame(
    profile,
    [
        "coluna",
        "tipo_dado",
        "total_registros",
        "qtd_preenchidos",
        "qtd_null",
        "qtd_vazio",
        "qtd_nao_preenchidos",
        "qtd_distintos",
        "pct_preenchido",
        "pct_null",
        "pct_vazio",
        "pct_distintos"
    ]
)

In [0]:
# ============================================================
# 7. EXIBIÇÃO
# ============================================================

# O QUE FAZ:
# Apresenta os resultados consolidados do Schema Profile para análise exploratória.

# COMO FAZ:
# Ordena as colunas pelo percentual de preenchimento, apresentando primeiro aquelas com menor completude.

# POR QUE É IMPORTANTE:
# Facilita a identificação rápida das colunas com maior ausência de dados e dos principais pontos de atenção.

# PERGUNTA RESPONDIDA:
# Quais colunas apresentam os maiores problemas de completude?

display(
    schema_profile.orderBy(
        F.col("pct_preenchido").asc()
    )
)

In [0]:
# ============================================================
# 8. VISUALIZAÇÃO - PREENCHIMENTO X NULL
# ============================================================

# O QUE FAZ:
# Apresenta visualmente a relação entre valores preenchidos e valores NULL existentes em cada coluna.

# COMO FAZ:
# Utiliza os percentuais de preenchimento e NULL calculados pelo Schema Profile para construir a visualização.

# POR QUE É IMPORTANTE:
# Permite identificar rapidamente quais colunas apresentam maior ausência de informação.

# PERGUNTA RESPONDIDA:
# Quais colunas apresentam maior ausência de dados?

schema_profile_preenchimento = (
    schema_profile
    .select(
        "coluna",
        "pct_preenchido",
        "pct_null"
    )
    .orderBy(
        F.col("pct_preenchido").asc()
    )
)

# Transformar para formato empilhado
schema_profile_stacked = (
    schema_profile_preenchimento
    .selectExpr(
        "coluna",
        "pct_preenchido as percentual",
        "'Preenchido' as tipo"
    )
    .union(
        schema_profile_preenchimento.selectExpr(
            "coluna",
            "pct_null as percentual",
            "'Nulo' as tipo"
        )
    )
)

display(
    schema_profile_stacked
)

Databricks visualization. Run in Databricks to view.

In [0]:
# ============================================================
# 9. VISUALIZAÇÃO - CARDINALIDADE
# ============================================================

# O QUE FAZ:
# Apresenta a proporção de valores distintos existentes em relação ao total de registros de cada coluna.

# COMO FAZ:
# Utiliza o percentual de valores distintos calculado durante a construção do Schema Profile.

# POR QUE É IMPORTANTE:
# A cardinalidade ajuda a identificar o comportamento estrutural das colunas, diferenciando campos com baixa e alta variabilidade.

# PERGUNTA RESPONDIDA:
# Quanto os valores de cada coluna variam dentro da tabela?

schema_profile_cardinalidade = (
    schema_profile
    .select(
        "coluna",
        "qtd_distintos",
        "pct_distintos"
    )
    .orderBy(
        F.col("pct_distintos").desc()
    )
)

display(
    schema_profile_cardinalidade
)

In [0]:
# ============================================================
# 10. VISUALIZAÇÃO - VALORES DISTINTOS
# ============================================================

# O QUE FAZ:
# Apresenta a quantidade absoluta de valores diferentes existentes em cada coluna.

# COMO FAZ:
# Utiliza a quantidade de valores distintos calculada durante o Schema Profile.

# POR QUE É IMPORTANTE:
# Permite avaliar a diversidade real dos dados e complementar a análise percentual de cardinalidade.

# PERGUNTA RESPONDIDA:
# Quantos valores diferentes existem em cada coluna?

schema_profile_distintos = (
    schema_profile
    .select(
        "coluna",
        "qtd_distintos"
    )
    .orderBy(
        F.col("qtd_distintos").desc()
    )
)

display(
    schema_profile_distintos
)

In [0]:
# ============================================================
# 11. CLASSIFICAÇÃO ESTRUTURAL DAS COLUNAS
# ============================================================

# O QUE FAZ:
# Classifica as colunas de acordo com o comportamento combinado de completude e cardinalidade.

# COMPLETUDE:
# Representa o percentual de registros que possuem um valor preenchido em determinada coluna.
# Uma completude alta indica que a maior parte dos registros possui informação disponível na coluna.
# Uma completude baixa indica uma quantidade significativa de registros sem informação, o que pode representar uma ausência esperada ou um possível problema de qualidade que deve ser investigado.

# CARDINALIDADE:
# Representa a proporção de valores distintos existentes em uma coluna em relação ao total de registros.
# Uma cardinalidade baixa indica que a coluna possui poucos valores diferentes e pode representar categorias,classificações ou domínios controlados. Uma cardinalidade alta indica grande diversidade de valores e pode representar identificadores, chaves, timestamps ou campos com alta variabilidade.

# COMO FAZ:
# Aplica regras sobre os percentuais de preenchimento e valores distintos para identificar diferentes perfis estruturais.

# POR QUE É IMPORTANTE:
# Permite transformar métricas isoladas em uma primeira interpretação sobre o comportamento de cada coluna e identificar campos que merecem investigação.

# PERGUNTA RESPONDIDA:
# Que comportamento estrutural cada coluna apresenta?

schema_profile_estrutura = (
    schema_profile
    .select(
        "coluna",
        "pct_preenchido",
        "pct_distintos"
    )
    .withColumn(
        "perfil_estrutural",

        F.when(
            (F.col("pct_preenchido") >= 95) &
            (F.col("pct_distintos") >= 80),
            "Alta completude / Alta cardinalidade"
        )

        .when(
            (F.col("pct_preenchido") >= 95) &
            (F.col("pct_distintos") < 10),
            "Alta completude / Baixa cardinalidade"
        )

        .when(
            (F.col("pct_preenchido") < 50) &
            (F.col("pct_distintos") < 10),
            "Baixa completude / Baixa cardinalidade"
        )

        .otherwise(
            "Comportamento intermediário"
        )
    )
)

display(
    schema_profile_estrutura
        .orderBy(
            F.col("pct_preenchido").asc()
        )
)


##### 03. DISTRIBUTION PROFILE — ANÁLISE DA DISTRIBUIÇÃO DOS DADOS

O Distribution Profile tem como objetivo analisar como os valores estão distribuídos dentro de cada coluna do DataFrame.

Enquanto o Schema Profile avalia características estruturais, como completude, valores NULL, valores vazios, quantidade de distintos e cardinalidade, esta etapa busca compreender a ***frequência e a concentração dos valores***.

###### O QUE É DISTRIBUIÇÃO?

Distribuição representa a forma como os valores de uma coluna estão distribuídos entre os registros da tabela.

Uma coluna pode apresentar 100% de preenchimento e possuir poucos valores responsáveis pela maior parte dos registros. Da mesma forma, outra coluna pode apresentar os mesmos 100% de preenchimento, mas possuir uma grande diversidade de valores com baixa concentração.

O Distribution Profile permite identificar essas diferenças de comportamento.

###### O QUE SERÁ ANALISADO?

Nesta etapa serão analisados:

- Frequência dos valores;
- Quantidade de valores distintos;
- Valores mais frequentes;
- Participação percentual dos valores mais frequentes;
- Nível de concentração dos valores;
- Distribuição dos valores entre os registros.

###### FREQUÊNCIA DOS VALORES

A frequência representa quantas vezes determinado valor aparece dentro de uma coluna.

Essa análise permite identificar valores predominantes e compreender quais informações representam a maior parte dos registros.

###### CONCENTRAÇÃO DOS VALORES

A concentração representa quanto dos registros está concentrado em determinados valores.

Uma coluna pode possuir poucos valores distintos e apresentar uma distribuição equilibrada entre eles. Da mesma forma, pode possuir poucos valores distintos e ter quase todos os registros concentrados em apenas um valor.

Essa diferença é relevante para compreender o comportamento real dos dados.

###### POR QUE É IMPORTANTE?

A análise de distribuição complementa o Schema Profile porque permite identificar comportamentos que não são evidentes apenas pela quantidade de valores preenchidos ou distintos.

Por exemplo, duas colunas podem apresentar 100% de completude, mas possuir comportamentos completamente diferentes:

***Coluna A***
- Poucos valores distintos;
- Alta concentração em um único valor.

***Coluna B***
- Muitos valores distintos;
- Baixa concentração em cada valor.

A análise de distribuição permite identificar essa diferença de comportamento.

###### ABORDAGEM GENÉRICA

Esta etapa não depende do significado das colunas e não utiliza regras específicas de negócio.

O framework não precisa saber se uma coluna representa cliente, produto, ocorrência, cidade, código, valor ou qualquer outro atributo.

A análise considera apenas o comportamento observado nos valores existentes no DataFrame.

###### RELAÇÃO COM O SCHEMA PROFILE

O Schema Profile responde principalmente:

***"Como está estruturada cada coluna?"***

O Distribution Profile complementa essa análise respondendo:

***"Como os valores estão distribuídos dentro de cada coluna?"***

###### PERGUNTA PRINCIPAL

***Como os valores estão distribuídos e concentrados dentro de cada coluna?***

###### PRÓXIMA ETAPA

Os resultados desta análise serão utilizados como base para o ***03. PATTERN PROFILE***, que irá aprofundar a análise sobre os padrões estruturais presentes nos valores, mantendo o framework independente do significado das colunas.

In [0]:
# ============================================================
# 12. RESUMO DA DISTRIBUIÇÃO
# ============================================================

# O QUE FAZ:
# Complementa o Schema Profile com informações relacionadas ao comportamento e à distribuição dos valores das colunas.

# COMO FAZ:
# Utiliza as informações já calculadas no Schema Profile e acrescenta métricas específicas de distribuição.

# POR QUE É IMPORTANTE:
# Permite analisar não apenas quanto uma coluna está preenchida, mas também como seus valores estão distribuídos.

# PERGUNTA RESPONDIDA:
# Como os valores estão distribuídos dentro de cada coluna?


distribution_profile = (
    schema_profile
    .select(
        "coluna",
        "tipo_dado",
        "total_registros",
        "qtd_preenchidos",
        "qtd_distintos",
        "pct_distintos"
    )
)

display(
    distribution_profile
)

In [0]:
# ============================================================
# 13. FREQUÊNCIA DOS VALORES
# ============================================================

# O QUE FAZ:
# Calcula a quantidade de ocorrências de cada valor existente nas colunas do DataFrame.

# COMO FAZ:
# Agrupa os valores individualmente por coluna e contabiliza suas respectivas ocorrências.

# POR QUE É IMPORTANTE:
# Permite identificar quais valores aparecem com maior frequência e compreender o comportamento da distribuição de cada coluna.

# PERGUNTA RESPONDIDA:
# Quantas vezes cada valor aparece em cada coluna?


frequencias = []

for campo in df.schema.fields:

    nome_coluna = campo.name

    frequencia_coluna = (
        df
        .select(
            F.lit(nome_coluna).alias("coluna"),
            F.col(nome_coluna).cast("string").alias("valor")
        )
        .where(
            F.col(nome_coluna).isNotNull()
        )
        .groupBy(
            "coluna",
            "valor"
        )
        .count()
    )

    frequencias.append(
        frequencia_coluna
    )


frequencia_profile = frequencias[0]

for frequencia_coluna in frequencias[1:]:

    frequencia_profile = (
        frequencia_profile
        .unionByName(frequencia_coluna)
    )


display(
    frequencia_profile
)